### Extrayendo la Dirección de Compra

Tenemos los tickets de una tienda de electrónica. Primero, reciclaremos el código de la sesión anterior para encontrar los conjuntos frecuentes. Luego, inyectaremos el algoritmo de reglas de asociación para separar los antecedentes de los consecuentes.

In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. DATOS CRUDOS
tickets_tech = [
    ['Laptop', 'Funda', 'Ratón', 'Cable HDMI'],
    ['Laptop', 'Ratón', 'Monitor'],
    ['Ratón', 'Pilas', 'Teclado'],
    ['Laptop', 'Funda'],
    ['Ratón', 'Pilas'],
    ['Laptop', 'Funda', 'Ratón'],
    ['Monitor', 'Cable HDMI'],
    ['Laptop', 'Ratón']
]

# 2. TRANSFORMACIÓN (One-Hot)
te = TransactionEncoder()
matriz_bool = te.fit_transform(tickets_tech)
df_tech = pd.DataFrame(matriz_bool, columns=te.columns_)

display(df_tech)

,Cable HDMI,Funda,Laptop,Monitor,Pilas,Ratón,Teclado
0,True,True,True,False,False,True,False
1,False,False,True,True,False,True,False
2,False,False,False,False,True,True,True
3,False,True,True,False,False,False,False
4,False,False,False,False,True,True,False
5,False,True,True,False,False,True,False
6,True,False,False,True,False,False,False
7,False,False,True,False,False,True,False


In [2]:
# 3. CONJUNTOS FRECUENTES (La clase pasada)
itemsets = apriori(df_tech, min_support=0.3, use_colnames=True)

# 4. REGLAS DE ASOCIACIÓN (El tema de hoy)
# Usamos 'confidence' como métrica base para generar las direcciones
reglas = association_rules(itemsets, metric="confidence", min_threshold=0.5)

print("--- TODAS LAS REGLAS DESCUBIERTAS ---")
display(reglas[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

--- TODAS LAS REGLAS DESCUBIERTAS ---


,antecedents,consequents,support,confidence,lift
0,frozenset({Laptop}),frozenset({Funda}),0.375,0.600000,1.600000
1,frozenset({Funda}),frozenset({Laptop}),0.375,1.000000,1.600000
2,frozenset({Ratón}),frozenset({Laptop}),0.500,0.666667,1.066667
3,frozenset({Laptop}),frozenset({Ratón}),0.500,0.800000,1.066667


### Reto Analítico: Laptop vs Ratón

Observa tu tabla resultante. El algoritmo creó dos reglas distintas para los mismos artículos:
1. `(Laptop) -> (Ratón)`
2. `(Ratón) -> (Laptop)`

Filtremos la tabla usando Pandas para ver exclusivamente estas dos reglas y comparar su Confianza.

In [3]:
display(reglas)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({Laptop}),frozenset({Funda}),0.625,0.375,0.375,0.600000,1.600000,1.0,0.140625,1.5625,1.000000,0.600000,0.360000,0.800000
1,frozenset({Funda}),frozenset({Laptop}),0.375,0.625,0.375,1.000000,1.600000,1.0,0.140625,inf,0.600000,0.600000,1.000000,0.800000
2,frozenset({Ratón}),frozenset({Laptop}),0.750,0.625,0.500,0.666667,1.066667,1.0,0.031250,1.1250,0.250000,0.571429,0.111111,0.733333
3,frozenset({Laptop}),frozenset({Ratón}),0.625,0.750,0.500,0.800000,1.066667,1.0,0.031250,1.2500,0.166667,0.571429,0.200000,0.733333


In [4]:
# Convertimos los frozensets a strings temporales para poder filtrar fácilmente en Pandas
reglas['ant_str'] = reglas['antecedents'].apply(lambda x: ', '.join(list(x)))
reglas['con_str'] = reglas['consequents'].apply(lambda x: ', '.join(list(x)))

# Filtramos la batalla direccional
batalla_direccional = reglas[
    ((reglas['ant_str'] == 'Laptop') & (reglas['con_str'] == 'Ratón')) |
    ((reglas['ant_str'] == 'Ratón') & (reglas['con_str'] == 'Laptop'))
]

print("--- ANÁLISIS DIRECCIONAL (Laptop vs Ratón) ---")
display(batalla_direccional[['antecedents',  'consequents', 'support','confidence', 'lift']])

--- ANÁLISIS DIRECCIONAL (Laptop vs Ratón) ---


,antecedents,consequents,support,confidence,lift
2,frozenset({Ratón}),frozenset({Laptop}),0.5,0.666667,1.066667
3,frozenset({Laptop}),frozenset({Ratón}),0.5,0.800000,1.066667


### 🧠 Evaluación Ejecutiva (Responde aquí)

El gerente de la tienda quiere hacer una estrategia de "Venta Cruzada" (Cross-selling) en la zona de cajas registradoras. Tiene espacio físico limitado y te hace dos preguntas:

**1. Análisis de Confianza:**
Basado en los resultados de tu `batalla_direccional`, ¿qué estrategia matemática es correcta?
*   Opción A: Poner exhibidores de Ratones junto a las Laptops.
*   Opción B: Poner exhibidores de Laptops junto a los Ratones.
> **Justifica usando el porcentaje exacto de Confianza:** [Escribe tu justificación aquí]

**2. Análisis de Lift:**
Observa que el valor de *Lift* es exactamente el mismo para `Laptop -> Ratón` que para `Ratón -> Laptop`. ¿Por qué ocurre esto matemáticamente? (Pista: Revisa la fórmula del Lift de la sesión pasada).
> **Tu Respuesta:** [Escribe tu justificación matemática aquí]

In [7]:
import random
import math

def generar_equipos(lista_alumnos, group_size = 5):
    alumnos_mezclados = lista_alumnos[:]
    random.shuffle(alumnos_mezclados)
    
    n = len(alumnos_mezclados)
    if n == 0:
        return []
        
    num_grupos = math.ceil(n / group_size)
    
    tamanos_grupos = [n // num_grupos + (1 if i < n % num_grupos else 0) for i in range(num_grupos)]
    
    equipos = []
    indice = 0
    
    for tamano in tamanos_grupos:
        equipos.append(alumnos_mezclados[indice:indice + tamano])
        indice += tamano
        
    return equipos
#"Linette","Angela",
alumnos = [ "Rogelio", "Juan Pablo", "Octavio", "Cristobal",
"Ramiro", "Diego Angel", "Emiliano", "Carlos Miguel",
"Andrea", "Rhamses", "Mayrin", "Christopher", "Kamila","Andres", "Diego Torres",
"Raul", "Ximena", "Luis", "Diego Lopez", "Juan Manuel", "Erik"]
equipos_formados = generar_equipos(alumnos,2)

for i, equipo in enumerate(equipos_formados, 1):
    print(f"Equipo {i} ({len(equipo)} integrantes): {', '.join(equipo)}")

Equipo 1 (2 integrantes): Raul, Octavio
Equipo 2 (2 integrantes): Cristobal, Diego Angel
Equipo 3 (2 integrantes): Ximena, Rogelio
Equipo 4 (2 integrantes): Diego Torres, Diego Lopez
Equipo 5 (2 integrantes): Carlos Miguel, Juan Pablo
Equipo 6 (2 integrantes): Emiliano, Mayrin
Equipo 7 (2 integrantes): Christopher, Luis
Equipo 8 (2 integrantes): Juan Manuel, Rhamses
Equipo 9 (2 integrantes): Kamila, Erik
Equipo 10 (2 integrantes): Andrea, Ramiro
Equipo 11 (1 integrantes): Andres
